In [1]:
#!pip3 install --upgrade git+https://github.com/Xilinx/DPU-PYNQ.git@design_contest_3.5 --no-build-isolation
#!pip3 install pynq-dpu --no-build-isolation
#!pip3 install git+https://github.com/voxelmorph/voxelmorph.git
# !pip install --no-cache-dir --upgrade "numpy<2"

In [2]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pynq import allocate
from pynq_dpu import DpuOverlay 
import xir
import vart
import time
import pynq

In [3]:
DPU_BITFILE = "dpu.bit"
XMODEL_FILE = "../xmodels/vxm2d_kv260_3.xmodel"
SAMPLE_INPUT = "../calibration_dataset/calib_pair_0.npy"

In [4]:
def check_file(path):
    if not os.path.isfile(path):
        print(f"ERROR: required file not found: {os.path.abspath(path)}", file=sys.stderr)
        sys.exit(1)

for p in (XMODEL_FILE, SAMPLE_INPUT):
    check_file(p)

In [5]:
overlay = DpuOverlay(DPU_BITFILE)
overlay.download()

In [6]:
graph = xir.Graph.deserialize(XMODEL_FILE)
root = graph.get_root_subgraph()
quant_params = {}

def get_all_device_subgraphs(parent):
    subs = []
    if parent.has_attr("device"):
        subs.append(parent)
    for c in parent.get_children():
        subs += get_all_device_subgraphs(c)
    return subs

all_subgraphs = get_all_device_subgraphs(root)
print("Device subgraph execution order:")
for idx, sg in enumerate(all_subgraphs):
    dev  = sg.get_attr("device")
    ins  = [t.name for t in sg.get_input_tensors()]
    outs = [t.name for t in sg.get_output_tensors()]
    print(f"[{idx:02d}] {sg.get_name():<50} type={dev} inputs={ins} outputs={outs}")

Device subgraph execution order:
[00] subgraph_input_24                                  type=USER inputs=[] outputs=['quant_input_24']
[01] subgraph_flow                                      type=CPU inputs=['quant_leaky_re_lu_118_fix', 'quant_input_23'] outputs=['spatial_transformer_4']
[02] subgraph_input_23                                  type=USER inputs=[] outputs=['quant_input_23']
[03] subgraph_quant_concatenate_55                      type=DPU inputs=['quant_input_23', 'quant_input_24'] outputs=['quant_leaky_re_lu_118_fix']


W0628 21:21:38.843015 590693 tool_function.cpp:171] [UNILOG][WARNING] The operator named flow, type: Lambda, is not defined in XIR. XIR creates the definition of this operator automatically. You should specify the shape and the data_type of the output tensor of this operation by set_attr("shape", std::vector<int>) and set_attr("data_type", std::string)
W0628 21:21:38.843498 590693 tool_function.cpp:171] [UNILOG][WARNING] The operator named spatial_transformer_4, type: SpatialTransformer, is not defined in XIR. XIR creates the definition of this operator automatically. You should specify the shape and the data_type of the output tensor of this operation by set_attr("shape", std::vector<int>) and set_attr("data_type", std::string)


In [7]:
def relu(x, alpha=26/256):           return np.where(x >= 0, x, alpha * x)
def identity(x):       return x
def fix2float(x):      return x.astype(np.float32)
def save_for_offline(pad_int, moving, i):
    outdir = '/home/root/jupyter_notebooks/CNN2FPGA/spatial_transform_flow/results'
    os.makedirs(outdir, exist_ok=True)
    np.save(f'{outdir}/moving{i}.npy',    moving)
    np.save(f'{outdir}/pad{i}.npy',    pad_int)
    return moving



def save_model(x,i):
    np.save(f'/home/root/jupyter_notebooks/CNN2FPGA/spatial_transform_flow/results/fixed{i}.npy', x)
    return x

cpu_op_map = {
    "activation": relu,
    "fix":        fix2float,
    "subgraph_input_24": save_model
}

runners    = []
cpu_funcs  = []
for sg in all_subgraphs:
    dev = sg.get_attr("device")
    print(sg.get_name())
    if dev == "DPU":
        runner = vart.Runner.create_runner(sg, "run")
 
        out_t = next(iter(sg.get_output_tensors()))

        fix_point = out_t.get_attr("fix_point")
 
        scale = 1.0 / (2 ** fix_point)
        zp    = 0

        quant_params[out_t.name] = (scale, zp)
        runners.append(runner)
        cpu_funcs.append(None)
    elif dev == "CPU" and sg.get_name().lower() == "subgraph_flow":
        runners.append(None)
        cpu_funcs.append(save_for_offline)
    else:
        runners.append(None)
        name = sg.get_name()
        func = next((cpu_op_map[k] for k in cpu_op_map if k in name), identity)
        cpu_funcs.append(func)

subgraph_input_24
subgraph_flow
subgraph_input_23
subgraph_quant_concatenate_55


In [8]:
def run_dpu_runner(runner, inputs):
    in_tensors  = runner.get_input_tensors()
    out_tensors = runner.get_output_tensors()
    in_bufs = []
    if isinstance(inputs, (list, tuple)):
        for arr, t in zip(inputs, in_tensors):
            buf = allocate(shape=tuple(t.dims), dtype=np.float32)
            np.copyto(buf, arr)
            in_bufs.append(buf)
    else:
        buf = allocate(shape=tuple(in_tensors[0].dims), dtype=np.float32)
        np.copyto(buf, inputs)
        in_bufs = [buf]
        
    out_bufs = [allocate(shape=tuple(t.dims), dtype=np.float32) for t in out_tensors]

    job_id = runner.execute_async(in_bufs, out_bufs)
    runner.wait(job_id)
    return [np.array(b) for b in out_bufs]

In [9]:
n_samples=20
data_dir='../calibration_dataset'
latency=[]
for i in range(5):
    t_list = []
    for n in range(n_samples):
        data = np.load(f'{data_dir}/calib_pair_{n}.npy', allow_pickle=True)
        moving = data[0].astype(np.float32)
        fixed  = data[1].astype(np.float32)

        moving_b = moving[None, ...]
        fixed_b  = fixed[None, ...]

        tensor_map = {
            'quant_input_23': moving_b,
            'quant_input_24': fixed_b
        }
        
        pending = set(range(len(all_subgraphs)))
        t0 = time.perf_counter()
        while pending:
            progress = False

            for i in list(pending):
                sg   = all_subgraphs[i]
                name = sg.get_name()
                dev  = sg.get_attr("device")

                insh  = sg.get_input_tensors()
                outsh = sorted(sg.get_output_tensors(), key=lambda t: t.name)

                if not all(t.name in tensor_map for t in insh):
                    continue

                inputs = [tensor_map[t.name] for t in insh]

                if dev == "DPU":
                    outputs = run_dpu_runner(runners[i], inputs)
                elif dev == "CPU":
                    break
                elif dev == "USER":
                    continue
            break

        t1 = time.perf_counter()
        t_list.append((t1 - t0) * 1000)

    ts = np.array(t_list)
    latency.append(ts)
    print(f'avrg latency: {ts.mean():.2f} ms ±{ts.std():.2f} ms')

latency= np.array(latency)
print(f'total latency: {latency.mean():.2f} ms ±{latency.std():.2f} ms')
    


avrg latency: 51.63 ms ±0.71 ms
avrg latency: 51.59 ms ±0.49 ms
avrg latency: 51.46 ms ±0.16 ms
avrg latency: 51.48 ms ±0.22 ms
avrg latency: 51.47 ms ±0.24 ms
total latency: 51.53 ms ±0.42 ms
